baseline code 


In [4]:
# -*- coding: utf-8 -*-
# NOTEBOOK : TEST BASELINE "NONE" (SANS ÉQUILIBRAGE) SUR 94 FEATURES

# =============================================================================
# CELLULE 1 : IMPORTS ET CONFIGURATION
# =============================================================================
print("--- Cellule 1: Imports et Configuration ---")

# --- Imports ---
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter # Ajouté pour la distribution de la cible

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, average_precision_score # average_precision_score est bon pour AUC PR
)

# LightGBM
try:
    import lightgbm as lgb
    lgbm_available = True
except ImportError:
    lgbm_available = False
    print("Warning: LightGBM non trouvé. Il sera ignoré si inclus dans les modèles.")

# --- Configuration des Chemins et Variables ---
DATA_PATH = "/home/kaoutar/Desktop/data/"
# Assurez-vous que ce fichier contient les 94 features finales assainies et enrichies
INPUT_FILE = os.path.join(DATA_PATH, 'dataset_X94_clean_enhanced.csv') # Vérifiez ce nom !

TARGET_COL_NAME = 'appetent' # C'est bien le nom de votre cible dans ce fichier
RANDOM_STATE = 42
# ---------------------------------------------

print("-" * 50)
print("Imports et Configuration de Base Terminés.")
print(f"Utilisation du fichier d'entrée : {INPUT_FILE}")
print(f"Colonne Cible Attendue : {TARGET_COL_NAME}")
print("Aucune méthode d'équilibrage explicite ne sera appliquée (test 'NONE').")
print("-" * 50)

# --- FIN Cellule 1 ---

--- Cellule 1: Imports et Configuration ---


--------------------------------------------------
Imports et Configuration de Base Terminés.
Utilisation du fichier d'entrée : /home/kaoutar/Desktop/data/dataset_X94_clean_enhanced.csv
Colonne Cible Attendue : appetent
Aucune méthode d'équilibrage explicite ne sera appliquée (test 'NONE').
--------------------------------------------------


In [5]:
# =============================================================================
# CELLULE 2 : CHARGEMENT DES DONNÉES (94 features) ET PRÉPARATION X, y
# =============================================================================
print(f"\n--- Chargement du jeu de données: {INPUT_FILE} ---")

X = None
y = None
data_loaded_successfully = False

if os.path.exists(INPUT_FILE):
    try:
        df_final_94_features = pd.read_csv(INPUT_FILE, sep=',')
        print(f"Données chargées. Shape initiale: {df_final_94_features.shape}")

        if TARGET_COL_NAME not in df_final_94_features.columns:
             raise ValueError(f"Colonne cible '{TARGET_COL_NAME}' manquante dans {INPUT_FILE}.")

        X = df_final_94_features.drop(TARGET_COL_NAME, axis=1)
        y = df_final_94_features[TARGET_COL_NAME]

        # S'assurer que X ne contient que des colonnes numériques
        X = X.select_dtypes(include=[np.number])
        if X.shape[1] != 94: # Ou le nombre exact attendu
            print(f"ATTENTION: Le nombre de features numériques dans X ({X.shape[1]}) pourrait ne pas être 94. Vérifiez.")

        if X.isna().sum().sum() > 0:
            print(f"ATTENTION: NaNs détectés dans X. Remplissage par 0...")
            X = X.fillna(0)
        if y.isna().any():
            print(f"ATTENTION: NaNs détectés dans y. Suppression des lignes...")
            valid_indices = ~y.isna(); X = X.loc[valid_indices]; y = y.loc[valid_indices]

        if not pd.api.types.is_numeric_dtype(y): y = y.astype(int)
        unique_y_values = np.unique(y)
        if not set(unique_y_values).issubset({0, 1}):
            raise ValueError(f"Cible non binaire après conversion: {unique_y_values}")

        # Pas besoin de filtre de variance si X_final_ready est déjà le résultat d'un tel filtre.
        # Si vous voulez être sûr, vous pouvez le remettre ici.

        print(f"\nPréparation X/y OK. Shape X final: {X.shape}, Shape y: {y.shape}")
        print(f"Distribution de la cible y: \n{y.value_counts(normalize=True)}")
        data_loaded_successfully = True

    except Exception as e:
        print(f"ERREUR lors du chargement ou préparation initiale: {e}")
else:
    print(f"ERREUR: Fichier {INPUT_FILE} non trouvé.")

# --- FIN Cellule 2 ---


--- Chargement du jeu de données: /home/kaoutar/Desktop/data/dataset_X94_clean_enhanced.csv ---
Données chargées. Shape initiale: (1051281, 95)

Préparation X/y OK. Shape X final: (1051281, 94), Shape y: (1051281,)
Distribution de la cible y: 
appetent
0    0.923968
1    0.076032
Name: proportion, dtype: float64


In [7]:
# =============================================================================
# CELLULE 3 : ÉTAPE V - PRÉPARATION AVANT MODÉLISATION (SANS ÉQUILIBRAGE)
# =============================================================================

X_train_scaled, X_test_scaled, y_train, y_test = None, None, None, None
data_ready_for_modeling = False

if data_loaded_successfully and X is not None and y is not None and not X.empty:
    print("\n--- ÉTAPE V: Préparation Avant Modélisation (Sans Équilibrage Explicite) ---")

    # 1. Séparation Train/Test
    print("Séparation Train/Test (80/20)...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y)
    print(f"  Shapes - Train: {X_train.shape}, Test: {X_test.shape}")

    # 2. PAS DE GESTION DU DÉSÉQUILIBRE EXPLICITE
    print("\nPas de rééchantillonnage appliqué (les modèles seront entraînés sur y_train original).")
    y_train_final_for_model = y_train.copy() # Cible pour l'entraînement (déséquilibrée)
    X_train_for_scaling = X_train.copy()     # Features d'entraînement (déséquilibrées)

    # 3. Mise à l'échelle (Scaling)
    print("\nMise à l'échelle (StandardScaler)...")
    numerical_cols = X.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X.select_dtypes(include='int').columns if X[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in X.columns if c in numerical_cols and c not in cols_to_exclude_from_scaling and c in X_train_for_scaling.columns]

    scaler = StandardScaler()
    X_train_scaled = X_train_for_scaling.copy()
    X_test_scaled = X_test.copy()

    if numerical_cols_to_scale:
         X_train_num_part = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_test_num_part = X_test_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_num_part)
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_num_part)
         print("  Scaling appliqué.")
    else:
         print("  Aucune colonne appropriée à scaler n'a été trouvée.")
    data_ready_for_modeling = True
else:
    print("Erreur: Données X ou y non chargées ou X vide. Arrêt avant Étape V.")

# --- FIN Cellule 3 ---


--- ÉTAPE V: Préparation Avant Modélisation (Sans Équilibrage Explicite) ---
Séparation Train/Test (80/20)...
  Shapes - Train: (841024, 94), Test: (210257, 94)

Pas de rééchantillonnage appliqué (les modèles seront entraînés sur y_train original).

Mise à l'échelle (StandardScaler)...
  Scaling appliqué.


In [8]:
# =============================================================================
# CELLULE 4 : ÉTAPE VI - ENTRAÎNEMENT ET ÉVALUATION (SANS ÉQUILIBRAGE EXPLICITE)
# =============================================================================

if data_ready_for_modeling and X_train_scaled is not None and y_train_final_for_model is not None:
    print(f"\n--- ÉTAPE VI: Entraînement et Évaluation (Sur Données Originales Déséquilibrées - 'NONE') ---")

    # Définir les modèles à tester (SANS class_weight='balanced')
    model_configs_none = {
        "RandomForest_NONE": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5),
        "LightGBM_NONE": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1) if lgbm_available else None
        # Vous pouvez ajouter ici LogisticRegression et GradientBoosting si vous voulez aussi leur baseline "NONE"
        # "LogisticRegression_NONE": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'),
        # "GradientBoosting_NONE": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5)
    }

    results_none = {} # Pour stocker les résultats de cette exécution "NONE"

    # Boucle pour entraîner et évaluer
    for name, model in model_configs_none.items():
        if model is None: continue
        print(f"\n--- Modèle : {name} ---")
        try:
            start_time = datetime.now()
            print(f"  Entraînement {name}...")
            model.fit(X_train_scaled.fillna(0), y_train_final_for_model)
            train_time = datetime.now() - start_time
            print(f"  Entraînement terminé en {train_time}")

            print(f"  Évaluation {name} sur Test Set...")
            X_test_eval = X_test_scaled.fillna(0)
            y_pred_test = model.predict(X_test_eval)
            y_proba_test = model.predict_proba(X_test_eval)[:, 1]

            print(f"\n  Classification Report (Test - {name}):")
            print(classification_report(y_test, y_pred_test, zero_division=0))
            # print(f"\n  Matrice de Confusion (Test - {name}):") # Optionnel
            # cm_model = confusion_matrix(y_test, y_pred_test); print(cm_model)

            roc_auc = roc_auc_score(y_test, y_proba_test) if len(np.unique(y_test)) > 1 else 0.5
            pr_auc = average_precision_score(y_test, y_proba_test) # Utilisation directe d'average_precision_score
            report_dict = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)

            print(f"\n  AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
            results_none[name] = {
                 'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                 'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                 'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                 'Precision_Class1': report_dict.get('1', {}).get('precision', 0)
            }
        except Exception as e:
            print(f"  ERREUR pour {name}: {e}")
            results_none[name] = {'AUC_ROC': np.nan, 'AUC_PR': np.nan, 'F1_Class1': np.nan}

    if len(results_none) > 0:
        print("\n\n--- Résultats Finaux pour Méthode d'Équilibrage: NONE ---")
        results_df_none = pd.DataFrame(results_none).T
        pd.options.display.float_format = '{:.4f}'.format
        print(results_df_none)

else:
    print("\nErreur: Données non prêtes pour l'étape VI.")
# --- FIN Cellule 4 ---


--- ÉTAPE VI: Entraînement et Évaluation (Sur Données Originales Déséquilibrées - 'NONE') ---

--- Modèle : RandomForest_NONE ---
  Entraînement RandomForest_NONE...
  Entraînement terminé en 0:02:27.551526
  Évaluation RandomForest_NONE sur Test Set...

  Classification Report (Test - RandomForest_NONE):
              precision    recall  f1-score   support

           0       0.94      0.99      0.97    194271
           1       0.74      0.30      0.42     15986

    accuracy                           0.94    210257
   macro avg       0.84      0.64      0.70    210257
weighted avg       0.93      0.94      0.93    210257


  AUC ROC: 0.9253, AUC PR: 0.5796

--- Modèle : LightGBM_NONE ---
  Entraînement LightGBM_NONE...
  Entraînement terminé en 0:00:13.295745
  Évaluation LightGBM_NONE sur Test Set...

  Classification Report (Test - LightGBM_NONE):
              precision    recall  f1-score   support

           0       0.95      0.99      0.97    194271
           1       0.70 